In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Data Preparation

In [ ]:
import os
import glob
from sklearn.model_selection import train_test_split
import nibabel as nib
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

DATA_DIR = "/kaggle/input/3d-liver-segmentation/Task03_Liver_rs" 
IMAGES_DIR = os.path.join(DATA_DIR, 'imagesTr')
LABELS_DIR = os.path.join(DATA_DIR, 'labelsTr')

image_files = sorted(glob.glob(os.path.join(IMAGES_DIR, '*.nii')))
label_files = sorted(glob.glob(os.path.join(LABELS_DIR, '*.nii')))

print(f"Found {len(image_files)} images and {len(label_files)} labels.")

train_img_paths, val_img_paths, train_lbl_paths, val_lbl_paths = train_test_split(
    image_files, label_files, 
    train_size=100, 
    test_size=23, 
    random_state=12, 
    shuffle=True
)

print(f"Training volumes: {len(train_img_paths)}")
print(f"Validation volumes: {len(val_img_paths)}")

In [ ]:
import nibabel as nib
import pandas as pd
import os

# Create a list to store the stats
data_stats = []

print("Scanning file headers... please wait.")

for i, (img_path, lbl_path) in enumerate(zip(image_files, label_files)):
    # Load only the header (fast)
    img_obj = nib.load(img_path)
    lbl_obj = nib.load(lbl_path)
    
    # Get dimensions
    img_shape = img_obj.shape
    lbl_shape = lbl_obj.shape
    
    # Check if image and label match (sanity check)
    match = (img_shape == lbl_shape)
    
    data_stats.append({
        "Index": i,
        "Filename": os.path.basename(img_path),
        "Shape": img_shape,
        "Label Match": match
    })

# Convert to a DataFrame for a nice table
df = pd.DataFrame(data_stats)

# --- DISPLAY STATISTICS ---
print("\n--- DATASET DIMENSION REPORT ---")
print(f"Total Volumes: {len(df)}")

# Separate dimensions into columns
dims = pd.DataFrame(df['Shape'].tolist(), columns=['Height', 'Width', 'Depth'])

print("\n--- RESOLUTION STATS ---")
print(f"Min Shape: {dims.min().values}")
print(f"Max Shape: {dims.max().values}")
print(f"Avg Shape: {dims.mean().values.astype(int)}")

print("\n--- SAMPLE OF 5 FILES ---")
print(df[['Filename', 'Shape']].head())

# Data Preprocessing

In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import nibabel as nib
import numpy as np
import random
import cv2  

class LiverResizeDataset(Dataset):
    def __init__(self, image_paths, label_paths, target_shape=(64, 128, 128), augment=False):
        self.image_paths = image_paths
        self.label_paths = label_paths
        self.target_shape = target_shape
        self.augment = augment 

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # 1. Load
        img_nii = nib.load(self.image_paths[idx])
        lbl_nii = nib.load(self.label_paths[idx])
        img_data = img_nii.get_fdata()
        lbl_data = lbl_nii.get_fdata()

        # --- CONCEPT APPLICATION: Contrast Enhancement & Spatial Filtering ---
        # Clip to Liver Window
        img_data = np.clip(img_data, -100, 400)
        
        # Normalize to 0-255 (uint8) for OpenCV
        img_min, img_max = img_data.min(), img_data.max()
        if img_max > img_min:
            img_data = 255 * (img_data - img_min) / (img_max - img_min)
        img_data = img_data.astype(np.uint8)

        processed_slices = []
        
        # Initialize CLAHE (Contrast Enhancement)
        # Kept as CLAHE per your request, using standard medical parameters
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))

        for i in range(img_data.shape[2]): # Iterate over depth (Z-axis)
            slice_img = img_data[:, :, i]

            # 1. Spatial Filtering (Gaussian Blur) 
            slice_img = cv2.GaussianBlur(slice_img, (3, 3), 0)

            # 2. Contrast Enhancement (CLAHE)
            slice_img = clahe.apply(slice_img)
            
            processed_slices.append(slice_img)
        
        # Stack back to 3D and convert back to float
        img_data = np.stack(processed_slices, axis=2).astype(np.float32)

        # Final Standardization
        mean = np.mean(img_data)
        std = np.std(img_data)
        img_data = (img_data - mean) / (std + 1e-8)
        # -------------------------------------------------------------------

        # 3. To Tensor
        img_tensor = torch.tensor(img_data, dtype=torch.float32).permute(2, 0, 1)
        lbl_tensor = torch.tensor(lbl_data, dtype=torch.float32).permute(2, 0, 1)

        # 4. DATA AUGMENTATION
        if self.augment:
            # Intensity Augmentation
            brightness_factor = random.uniform(0.8, 1.2)
            contrast_factor = random.uniform(0.8, 1.2)
            img_tensor = (img_tensor * contrast_factor) * brightness_factor
            
            # Geometric Augmentations
            if random.random() > 0.5:
                img_tensor = torch.flip(img_tensor, dims=[2])
                lbl_tensor = torch.flip(lbl_tensor, dims=[2])
            
            if random.random() > 0.5:
                img_tensor = torch.flip(img_tensor, dims=[1])
                lbl_tensor = torch.flip(lbl_tensor, dims=[1])

            if random.random() > 0.5:
                k = random.randint(1, 3) 
                img_tensor = torch.rot90(img_tensor, k, dims=[1, 2])
                lbl_tensor = torch.rot90(lbl_tensor, k, dims=[1, 2])

        # 5. Add Dims & Resize
        img_tensor = img_tensor.unsqueeze(0).unsqueeze(0)
        lbl_tensor = lbl_tensor.unsqueeze(0).unsqueeze(0)

        img_resized = F.interpolate(img_tensor, size=self.target_shape, mode='trilinear', align_corners=False)
        lbl_resized = F.interpolate(lbl_tensor, size=self.target_shape, mode='nearest')

        return img_resized.squeeze(0), lbl_resized.squeeze(0)

# --- RE-INITIALIZE LOADERS ---
train_ds = LiverResizeDataset(train_img_paths, train_lbl_paths, augment=True)
val_ds = LiverResizeDataset(val_img_paths, val_lbl_paths, augment=False)

train_loader = DataLoader(train_ds, batch_size=2, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=2, shuffle=False)

print("Data Augmentation + Spatial Filtering + Contrast Enhancement enabled.")

# Visualization

In [ ]:
import matplotlib.pyplot as plt
import random
import numpy as np
import nibabel as nib

def visualize_train_vs_val(train_ds, val_ds, samples_per_split=2):
    """
    Picks random samples from Train and Val and plots them side-by-side.
    """
    total_rows = samples_per_split
    fig, axes = plt.subplots(total_rows, 2, figsize=(12, 5 * total_rows))
    
    print(f"Visualizing {samples_per_split} samples from TRAIN and {samples_per_split} from VAL...\\n")

    # Helper function to process and plot a single row
    def plot_row(dataset, row_idx, split_name):
        # 1. Pick random index
        idx = random.randint(0, len(dataset)-1)
        
        # 2. Get Raw Data (Before)
        raw_path = dataset.image_paths[idx]
        raw_nii = nib.load(raw_path)
        raw_data = raw_nii.get_fdata()
        raw_mid = raw_data.shape[2] // 2
        
        # 3. Get Processed Data (After)
        processed_img, processed_lbl = dataset[idx]
        proc_data = processed_img.squeeze().numpy()
        proc_mid = proc_data.shape[0] // 2
        
        # 4. PLOT
        # Column 0: BEFORE (Raw)
        axes[row_idx, 0].imshow(np.rot90(raw_data[:, :, raw_mid]), cmap='gray')
        axes[row_idx, 0].set_title(f"[{split_name}] Patient {idx} - BEFORE\nRaw Shape: {raw_data.shape}")
        axes[row_idx, 0].axis('off')
        
        # Column 1: AFTER (Processed)
        # Note: This 'Processed' image now includes your Blur + CLAHE
        axes[row_idx, 1].imshow(np.rot90(proc_data[proc_mid, :, :]), cmap='gray')
        axes[row_idx, 1].set_title(f"[{split_name}] Patient {idx} - AFTER (Blurred+CLAHE)\nTarget: {dataset.target_shape}")
        axes[row_idx, 1].axis('off')

    for i in range(samples_per_split):
        plot_row(train_ds, i, "TRAIN")

    plt.tight_layout()
    plt.show()

# Run it
visualize_train_vs_val(train_ds, val_ds, samples_per_split=2)

# U-net Defining

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# --- 1. THE MISSING HELPER BLOCK ---
class DoubleConv3D(nn.Module):
    """
    (Conv3D -> BatchNorm -> ReLU) * 2
    This keeps the image size constant because we use padding=1.
    """
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            # First Conv
            nn.Conv3d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True),
            # Second Conv
            nn.Conv3d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)

# --- 2. THE MAIN U-NET ---
class ClassicalUNet3D(nn.Module):
    def __init__(self, n_channels=1, n_classes=1):
        super().__init__()
        
        # --- ENCODER ---
        self.inc = DoubleConv3D(n_channels, 32)       
        self.down1 = DoubleConv3D(32, 64)             
        self.down2 = DoubleConv3D(64, 128)             
        self.down3 = DoubleConv3D(128, 256)
        self.down4 = DoubleConv3D(256, 512)
        
        # --- BOTTLENECK ---
        self.bottleneck = DoubleConv3D(512, 1024)
        
        # --- DECODER ---
        self.up1 = nn.ConvTranspose3d(1024, 512, kernel_size=2, stride=2)
        self.conv_up1 = DoubleConv3D(1024, 512) 
        
        self.up2 = nn.ConvTranspose3d(512, 256, kernel_size=2, stride=2)
        self.conv_up2 = DoubleConv3D(512, 256)
        
        self.up3 = nn.ConvTranspose3d(256,128 , kernel_size=2, stride=2)
        self.conv_up3 = DoubleConv3D(256,128)
        
        self.up4 = nn.ConvTranspose3d(128,64, kernel_size=2, stride=2)
        self.conv_up4 = DoubleConv3D(128,64)

        self.up5 = nn.ConvTranspose3d(64,32, kernel_size=2, stride=2)
        self.conv_up5 = DoubleConv3D(64,32)
        
        self.outc = nn.Conv3d(32, n_classes, kernel_size=1)

    def forward(self, x):
        # Encoder
        x1 = self.inc(x)
        x2 = self.down1(F.max_pool3d(x1, 2))
        x3 = self.down2(F.max_pool3d(x2, 2))
        x4 = self.down3(F.max_pool3d(x3, 2))
        x5 = self.down4(F.max_pool3d(x4, 2))
        
        # Bottleneck
        x_bot = self.bottleneck(F.max_pool3d(x5, 2))
        
        # Decoder
        
        # Up 1
        x = self.up1(x_bot)
        if x.shape != x5.shape:
             x = F.interpolate(x, size=x5.shape[2:], mode='trilinear', align_corners=False)
        x = torch.cat([x5, x], dim=1)
        x = self.conv_up1(x)
        
        # Up 2
        x = self.up2(x)
        if x.shape != x4.shape:
             x = F.interpolate(x, size=x4.shape[2:], mode='trilinear', align_corners=False)
        x = torch.cat([x4, x], dim=1)
        x = self.conv_up2(x)
        
        # Up 3
        x = self.up3(x)
        if x.shape != x3.shape:
             x = F.interpolate(x, size=x3.shape[2:], mode='trilinear', align_corners=False)
        x = torch.cat([x3, x], dim=1)
        x = self.conv_up3(x)

        # Up 4
        x = self.up4(x)
        if x.shape != x2.shape:
             x = F.interpolate(x, size=x2.shape[2:], mode='trilinear', align_corners=False)
        x = torch.cat([x2, x], dim=1)
        x = self.conv_up4(x)

        # Up 5
        x = self.up5(x)
        if x.shape != x1.shape:
             x = F.interpolate(x, size=x1.shape[2:], mode='trilinear', align_corners=False)
        x = torch.cat([x1, x], dim=1)
        x = self.conv_up5(x)
        
        return self.outc(x)

# Re-initialize the model to ensure it is clean
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ClassicalUNet3D().to(device)
print("Model initialized successfully.")

# Training

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import time
import copy
import os
import sys

# --- 1. HELPERS ---
class EarlyStopping:
    def __init__(self, patience=7, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.best_loss = None 
        self.best_model_state = None 
        self.best_epoch = 0
        self.best_train_dice = None
        self.best_train_loss = None

    def __call__(self, val_dice, val_loss, train_dice, train_loss, model, epoch):
        score = val_dice
        
        if self.best_score is None:
            self.save_best(val_dice, val_loss, train_dice, train_loss, model, epoch)
        elif score < self.best_score + self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.save_best(val_dice, val_loss, train_dice, train_loss, model, epoch)
            self.counter = 0 

    def save_best(self, val_dice, val_loss, train_dice, train_loss, model, epoch):
        self.best_score = val_dice
        self.best_loss = val_loss
        self.best_train_dice = train_dice
        self.best_train_loss = train_loss
        self.best_model_state = copy.deepcopy(model.state_dict())
        self.best_epoch = epoch

def calculate_dice(logits, targets, smooth=1e-6):
    preds = (torch.sigmoid(logits) > 0.5).float()
    intersection = (preds * targets).sum()
    return (2. * intersection + smooth) / (preds.sum() + targets.sum() + smooth)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# --- 2. CONFIGURATION ---
LR = 0.0003
EPOCHS = 100 
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SAVE_PATH = "best_classical_unet.pth"

# --- 3. INITIALIZE MODEL ---
model = ClassicalUNet3D().to(device)
optimizer = optim.Adam(model.parameters(), lr=LR)

# *** CHANGED: BACK TO STANDARD BCE ***

criterion = nn.BCEWithLogitsLoss()

scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)
early_stopping = EarlyStopping()

# --- 4. START TRAINING ---
total_params = count_parameters(model)
print(f"Classical Model Parameters: {total_params:,}")
print(f"Starting Classical Training on {device} (BCE Only)...")
print("-" * 65)

total_start_time = time.time()
stop_epoch = EPOCHS 
prev_lr = LR 

for epoch in range(EPOCHS):
    epoch_start = time.time()
    
    # --- TRAIN ---
    model.train()
    train_loss = 0
    train_dice = 0
    total_batches = len(train_loader)
    
    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        
        # *** STANDARD BCE LOSS ***
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        d = calculate_dice(outputs, labels)
        train_loss += loss.item()
        train_dice += d.item()
        
        # Manual Bar
        percent = int(100 * (batch_idx + 1) / total_batches)
        bar_len = 20
        filled_len = int(bar_len * percent / 100)
        bar = '=' * filled_len + '-' * (bar_len - filled_len)
        sys.stdout.write(f"\rEpoch {epoch+1}/{EPOCHS} [{bar}] {percent}% | Loss: {loss.item():.4f} | Dice: {d.item():.4f}")
        sys.stdout.flush()

    # Wipe Line
    sys.stdout.write("\r" + " " * 85 + "\r")
    sys.stdout.flush()
    
    avg_train_loss = train_loss / len(train_loader)
    avg_train_dice = train_dice / len(train_loader)
    
    # --- VALIDATE ---
    model.eval()
    val_loss = 0
    val_dice = 0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            
            loss = criterion(outputs, labels)
            
            val_loss += loss.item()
            val_dice += calculate_dice(outputs, labels).item()
            
    avg_val_loss = val_loss / len(val_loader)
    avg_val_dice = val_dice / len(val_loader)
    
    # --- SUMMARY ---
    duration = time.time() - epoch_start
    print(f"Epoch {epoch+1}/{EPOCHS} | Time: {duration:.0f}s")
    print(f"   Train Loss: {avg_train_loss:.4f} | Train Dice: {avg_train_dice:.4f}")
    print(f"   Val Loss:   {avg_val_loss:.4f} | Val Dice:   {avg_val_dice:.4f}")
    
    # --- UPDATES ---
    scheduler.step(avg_val_dice)
    
    current_lr = optimizer.param_groups[0]['lr']
    if current_lr != prev_lr:
        print(f"   >>> Learning Rate reduced to {current_lr:.1e}")
        prev_lr = current_lr
    
    early_stopping(avg_val_dice, avg_val_loss, avg_train_dice, avg_train_loss, model, epoch+1)
    
    if early_stopping.early_stop:
        print(f"\nEarly stopping triggered at Epoch {epoch+1}.")
        stop_epoch = epoch + 1
        break
    
    print("-" * 65)

total_end_time = time.time()
total_duration = total_end_time - total_start_time

# --- 5. SAVE ---
if early_stopping.best_model_state is not None:
    torch.save(early_stopping.best_model_state, SAVE_PATH)
    print(f"\nTraining Finished. Best Model Saved to: {SAVE_PATH}")
else:
    print("\nWarning: No best model found.")

# --- 6. FINAL REPORT ---
print("\n" + "="*40)
print("FINAL CLASSICAL MODEL REPORT")
print("="*40)
print(f"Total Parameters:       {total_params:,}")
print(f"Total Training Time:    {total_duration:.2f} seconds")
print(f"Stopped at Epoch:       {stop_epoch}")
print(f"Best Model Found at:    Epoch {early_stopping.best_epoch}")
print(f"Final Learning Rate:    {optimizer.param_groups[0]['lr']:.1e}")
print("-" * 40)
print(f"Best Validation Dice:   {early_stopping.best_score:.4f}")
print(f"Best Validation Loss:   {early_stopping.best_loss:.4f}")
print(f"Best Train Dice:        {early_stopping.best_train_dice:.4f}")
print(f"Best Train Loss:        {early_stopping.best_train_loss:.4f}")
print("="*40)

# Evaluation

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix, 
    roc_curve, 
    auc, 
    precision_recall_fscore_support, 
    accuracy_score
)

def evaluate_model(model, val_loader, device):
    model.eval()
    y_true = []
    y_pred_probs = []
    
    print("Generating predictions for comprehensive evaluation...")
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model(images)
            probs = torch.sigmoid(outputs)
            
            y_true.append(labels.cpu().numpy().flatten())
            y_pred_probs.append(probs.cpu().numpy().flatten())

    y_true = np.concatenate(y_true)
    y_pred_probs = np.concatenate(y_pred_probs)
    
    y_true_bin = (y_true > 0).astype(int)
    y_pred_bin = (y_pred_probs > 0.5).astype(int)

    # Calculation of Metrics 
    precision, recall, f_score, _ = precision_recall_fscore_support(y_true_bin, y_pred_bin, average='binary')
    accuracy = accuracy_score(y_true_bin, y_pred_bin)
    
    # ROC/AUC Calculation
    fpr, tpr, _ = roc_curve(y_true_bin, y_pred_probs)
    roc_auc = auc(fpr, tpr)

    print("-" * 30)
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F-score:   {f_score:.4f}")
    print(f"AUC Score: {roc_auc:.4f}") 
    print("-" * 30)

    # Visualization: Confusion Matrix
    cm = confusion_matrix(y_true_bin, y_pred_bin)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title("Confusion Matrix (Liver vs. Background)")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()

    # Visualization: ROC Curve & AUC Shading
    plt.figure(figsize=(7, 6))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
    # This shades the area under the curve for "AUC Visualization"
    plt.fill_between(fpr, tpr, color='darkorange', alpha=0.2, label='AUC Area')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic (ROC)')
    plt.legend(loc="lower right")
    plt.grid(alpha=0.3)
    plt.show()

# Run the evaluation
evaluate_model(model, val_loader, device)


# Documentation

### 3D U-Net Architecture Details
Our implementation follows the U-Net philosophy which consists of a symmetrical 
contracting (encoder) and expansive (decoder) path.

1. **Contracting Path (Encoder)**: 
   - Uses repeated application of two 3x3x3 convolutions (DoubleConv3D). 
   - Each convolution is followed by a Batch Normalization layer and a ReLU activation.
   - 3D Max Pooling (2x2x2) is used for downsampling, doubling the number of feature channels at each step.

2. **Bottleneck**:
   - The lowest resolution level containing 1024 filters. It mediates between the encoder 
     and decoder, capturing the most complex, high-level features of the liver volume.

3. **Expansive Path (Decoder)**:
   - Uses 3D Transposed Convolutions (up-convolutions) to restore spatial resolution.
   - At each step, we concatenate the upsampled feature map with the corresponding 
     feature map from the encoder (Skip Connections).
   - This prevents the 'vanishing gradient' problem and allows the model to recover 
     fine-grained spatial information lost during downsampling.

4. **Output Layer**:
   - A final 1x1x1 convolution is used to map the 32 feature channels to the 
     desired number of classes (binary segmentation).

### Research References:
- **[2D Foundations]**: Ronneberger, O., Fischer, P., & Brox, T. (2015). 
  "U-Net: Convolutional Networks for Biomedical Image Segmentation." 
- **[3D Extension]**: Çiçek, Ö., et al. (2016). 
  "3D U-Net: Learning Dense Volumetric Segmentation from Sparse Annotation."

# Comparative Analysis

1. **Experimental Performance**:
   - The 3D U-Net is specifically compared to 2D architectures. While 2D U-Nets process 
     slices independently, our 3D model utilizes volumetric kernels to recognize 
     organ continuity across the Z-axis.

2. **Pros & Cons**:
   - **Pros**: Maintains 3D spatial context; reduces "staircase" artifacts in 
     volumetric reconstruction; highly effective for the Liver dataset.
   - **Cons**: Extremely high GPU memory (VRAM) consumption; requires lower batch sizes 
     and specific resizing (64x128x128) to remain computationally feasible.

3. **Justification**:
   - For liver segmentation, the organ's boundary is often hard to distinguish from 
     adjacent tissues in a single slice. The 3D context provided by this architecture 
     allows the model to use information from neighboring slices to verify organ boundaries.